In [ ]:
# ==============================================================================
# KAGGLE SETUP BLOCK (INDEPENDENT BLOCK)
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

# Detect Kaggle environment
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if IS_KAGGLE:
    print("--- Detected Kaggle Environment ---")
    
    # 1. Clone repository if not present
    REPO_URL = "https://github.com/HCMUS-VIR-Nhom8/DINOv3-FAISS-HNSW-SOP-VisualProductSearch.git"
    REPO_DIR = Path("/kaggle/working/DINOv3-FAISS-HNSW-SOP-VisualProductSearch")
    
    if not REPO_DIR.exists():
        print(f"Cloning repository: {REPO_URL}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    
    # Change working directory of the kernel to notebooks directory
    # so that PROJECT_ROOT = Path("..").resolve() works correctly
    NOTEBOOKS_DIR = REPO_DIR / "notebooks"
    if NOTEBOOKS_DIR.exists():
        os.chdir(str(NOTEBOOKS_DIR))
        print(f"Changed working directory to: {os.getcwd()}")
        
    # Add project root to sys.path so src imports work
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
        print("Added repository root to python path.")
        
    # 2. Download and unzip dataset from Google Drive
    DATA_DIR = REPO_DIR / "data" / "raw" / "Stanford_Online_Products"
    ZIP_PATH = REPO_DIR / "Stanford_Online_Products.zip"
    
    if not DATA_DIR.exists() or not (DATA_DIR / "Ebay_train.txt").exists():
        print("Dataset not found. Downloading from Google Drive...")
        
        # Install gdown if needed
        try:
            import gdown
        except ImportError:
            print("Installing gdown...")
            subprocess.run([sys.executable, "-m", "pip", "install", "gdown"], check=True)
            import gdown
            
        # Download Stanford Online Products ZIP
        file_id = "1TclrpQOF_ullUP99wk_gjGN8pKvtErG8"
        url = f"https://drive.google.com/uc?id={file_id}"
        print(f"Downloading from Google Drive ID: {file_id}")
        gdown.download(url, str(ZIP_PATH), quiet=False)
        
        # Extract the zip file
        print("Extracting dataset...")
        import zipfile
        raw_dir = REPO_DIR / "data" / "raw"
        raw_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(str(ZIP_PATH), 'r') as zip_ref:
            zip_ref.extractall(str(raw_dir))
            
        print("Dataset extraction completed.")
        
        # Clean up zip file
        if ZIP_PATH.exists():
            ZIP_PATH.unlink()
            print("Cleaned up ZIP file.")
            
    # Note on HF login for gated DINOv3
    print("\n--- Hugging Face Access Note ---")
    print("DINOv3 is a gated Hugging Face model. If access token is required, run:")
    print("from huggingface_hub import login; login(token='YOUR_HF_TOKEN')\n")
else:
    print("Running in local environment. Setup skipped.")


# Notebook 02 — Baseline Visual Product Search

## Mục tiêu

Notebook này hiện thực hóa **baseline retrieval** sau khi Notebook 01 đã tạo:

```text
data/sampled/sop_20k.csv
```

Baseline được cố định:

```text
Input image
    ↓
Resize
    ↓
Pretrained ResNet50
    ↓
Global Average Pooling
    ↓
L2 Normalization
    ↓
Exact Cosine Similarity Search
    ↓
Top-K retrieved products
```

Notebook được chia thành 2 phần:

### Offline

```text
Gallery images
    → preprocessing
    → ResNet50
    → embedding
    → L2
    → lưu gallery embeddings
```

### Online

```text
Query image
    → preprocessing
    → ResNet50
    → embedding
    → L2
    → exact cosine search trên gallery embeddings
    → Top-K
```

Sau đó notebook đo:

- Recall@1
- Recall@5
- Recall@10
- Recall@20
- Recall@50
- Recall@100
- mAP
- offline embedding time
- online query latency
- embedding/index memory
- qualitative retrieval

> **Lưu ý:** Notebook này không dùng DINOv3, HNSW hoặc metadata re-ranking. Đây là baseline thuần để làm mốc so sánh với proposed.

## Cell 1 — Configuration

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

# Input từ Notebook 01
SPLIT_DIR = PROJECT_ROOT / "data" / "sampled"
SAMPLE_FILE = SPLIT_DIR / "sop_20k.csv"

# Output baseline
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_FILE = OUTPUT_DIR / "gallery_embeddings.npy"
GALLERY_META_FILE = OUTPUT_DIR / "gallery_metadata.csv"
QUERY_RESULTS_FILE = OUTPUT_DIR / "retrieval_results.npy"
METRICS_FILE = OUTPUT_DIR / "metrics.json"
LATENCY_FILE = OUTPUT_DIR / "latency.json"

# Model
MODEL_NAME = "resnet50"
PRETRAINED = True

# Input
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# Retrieval
KS = [1, 5, 10, 20, 50, 100]

# Evaluation
MAX_QUERIES = None       # None = toàn bộ query
RANDOM_SEED = 42

# Device
DEVICE = "cuda"          # đổi thành "cpu" nếu cần

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SAMPLE_FILE :", SAMPLE_FILE)
print("OUTPUT_DIR  :", OUTPUT_DIR)

## Cell 2 — Imports

In [ ]:
import os
import sys
import json
import time
import platform
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import models, transforms

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Cell 3 — Kiểm tra input từ Notebook 01

In [ ]:
if not SAMPLE_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {SAMPLE_FILE}. "
        "Hãy chạy Notebook 01 trước."
    )

sample_df = pd.read_csv(SAMPLE_FILE)

print("Rows:", len(sample_df))
print("Classes:", sample_df["class_id"].nunique())

display(sample_df.head())

## Cell 4 — Kiểm tra image paths

In [ ]:
sample_df["exists"] = sample_df["image_path"].map(os.path.exists)

print("Existing:", sample_df["exists"].sum())
print("Missing :", (~sample_df["exists"]).sum())

if not sample_df["exists"].all():
    display(
        sample_df.loc[
            ~sample_df["exists"],
            ["image_id", "class_id", "image_path"]
        ].head(20)
    )

    raise FileNotFoundError(
        "Có image path không tồn tại."
    )

print("✓ All image paths exist.")

## Cell 5 — Tạo query/gallery split cho baseline

Baseline cần một **gallery** để search và một **query set** để evaluation.

Split này được tạo **sau sampling 20K**, không sampling lại từ 120K.

Với mỗi `class_id`:

- 80% → gallery
- 20% → query
- tối thiểu 1 query
- tối thiểu 1 gallery image

Điều này đảm bảo query có positive image trong gallery.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

gallery_parts = []
query_parts = []

for class_id, group in sample_df.groupby("class_id"):
    group = group.sample(
        frac=1,
        random_state=RANDOM_SEED + int(class_id) % 100000
    )

    n = len(group)

    n_query = max(1, int(round(n * 0.20)))
    n_query = min(n_query, n - 1)

    query_parts.append(group.iloc[:n_query])
    gallery_parts.append(group.iloc[n_query:])

query_df = pd.concat(
    query_parts,
    ignore_index=True
)

gallery_df = pd.concat(
    gallery_parts,
    ignore_index=True
)

print("Gallery:", len(gallery_df))
print("Query  :", len(query_df))

print("Gallery classes:", gallery_df.class_id.nunique())
print("Query classes  :", query_df.class_id.nunique())

missing_query_classes = (
    set(query_df.class_id)
    - set(gallery_df.class_id)
)

print(
    "Query classes missing in gallery:",
    len(missing_query_classes)
)

assert len(missing_query_classes) == 0

## Cell 6 — Lưu query/gallery split

In [ ]:
gallery_file = SPLIT_DIR / "baseline_gallery.csv"
query_file = SPLIT_DIR / "baseline_query.csv"

gallery_df.to_csv(
    gallery_file,
    index=False
)

query_df.to_csv(
    query_file,
    index=False
)

print("Saved:", gallery_file)
print("Saved:", query_file)

## Cell 7 — Kiểm tra split distribution

In [ ]:
split_stats = pd.DataFrame({
    "gallery": gallery_df["class_id"].value_counts(),
    "query": query_df["class_id"].value_counts(),
}).fillna(0)

split_stats["total"] = (
    split_stats["gallery"]
    + split_stats["query"]
)

display(split_stats.head(20))

print("Gallery ratio:", len(gallery_df) / len(sample_df))
print("Query ratio  :", len(query_df) / len(sample_df))

## Cell 8 — Baseline preprocessing

Baseline chỉ dùng preprocessing tối thiểu:

```text
PIL image
 → RGB
 → Resize(224,224)
 → ToTensor
 → ImageNet normalization
```

Không sử dụng:

- object detection
- segmentation
- background removal
- illumination correction
- DINO-specific preprocessing
- metadata

In [ ]:
IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

baseline_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ),
])

print(baseline_transform)

## Cell 9 — Preview preprocessing

In [ ]:
row = gallery_df.iloc[0]

original = Image.open(
    row["image_path"]
).convert("RGB")

processed_tensor = baseline_transform(original)

# Unnormalize for visualization
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

processed_image = (
    processed_tensor.cpu() * std + mean
).clamp(0, 1)

plt.figure(figsize=(9, 4))

ax = plt.subplot(1, 2, 1)
ax.imshow(original)
ax.set_title("Original")
ax.axis("off")

ax = plt.subplot(1, 2, 2)
ax.imshow(processed_image.permute(1, 2, 0))
ax.set_title("Baseline input: 224x224")
ax.axis("off")

plt.tight_layout()
plt.show()

## Cell 10 — Dataset class

In [ ]:
class SOPImageDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return {
            "image": image,
            "image_id": int(row["image_id"]),
            "class_id": int(row["class_id"]),
            "index": idx,
        }

## Cell 11 — DataLoader

In [ ]:
gallery_dataset = SOPImageDataset(
    gallery_df,
    transform=baseline_transform
)

query_dataset = SOPImageDataset(
    query_df,
    transform=baseline_transform
)

gallery_loader = DataLoader(
    gallery_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

query_loader = DataLoader(
    query_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

print("Gallery batches:",
      len(gallery_loader))

print("Query batches:",
      len(query_loader))

## Cell 12 — Load pretrained ResNet50

Baseline encoder:

```text
ResNet50 pretrained on ImageNet
        ↓
remove classification FC
        ↓
2048-dimensional feature
```

Vì `avgpool` nằm ngay trước `fc`, output sau `avgpool` là:

```text
[B, 2048, 1, 1]
```

sau flatten:

```text
[B, 2048]
```

In [ ]:
weights = (
    models.ResNet50_Weights.DEFAULT
    if PRETRAINED
    else None
)

resnet = models.resnet50(
    weights=weights
)

# Remove classification head
encoder = nn.Sequential(
    *list(resnet.children())[:-1]
)

encoder = encoder.to(DEVICE)
encoder.eval()

for parameter in encoder.parameters():
    parameter.requires_grad = False

print(encoder)

## Cell 13 — Kiểm tra feature dimension

In [ ]:
dummy = torch.randn(
    2, 3,
    IMAGE_SIZE,
    IMAGE_SIZE
).to(DEVICE)

with torch.inference_mode():
    feature = encoder(dummy)

print("Raw output shape:", feature.shape)

feature = torch.flatten(
    feature,
    start_dim=1
)

print("Flattened feature shape:", feature.shape)

FEATURE_DIM = feature.shape[1]

assert FEATURE_DIM == 2048

print("Feature dimension:", FEATURE_DIM)

## Cell 14 — Hàm L2 normalization

In [ ]:
def l2_normalize(x, eps=1e-12):
    """
    Row-wise L2 normalization.
    x: [N, D]
    """
    norm = torch.linalg.vector_norm(
        x,
        ord=2,
        dim=1,
        keepdim=True
    )

    return x / norm.clamp_min(eps)


test_x = torch.randn(4, FEATURE_DIM).to(DEVICE)
test_y = l2_normalize(test_x)

norms = torch.linalg.vector_norm(
    test_y,
    ord=2,
    dim=1
)

print(norms)

assert torch.allclose(
    norms,
    torch.ones_like(norms),
    atol=1e-5
)

print("✓ L2 normalization correct.")

## Cell 15 — Hàm extract embeddings

Đây là **offline feature extraction**.

Input:

```text
gallery images
```

Output:

```text
gallery_embeddings.npy
```

Shape:

```text
[num_gallery_images, 2048]
```

In [ ]:
def extract_embeddings(
    model,
    dataloader,
    device,
):
    model.eval()

    all_embeddings = []
    all_image_ids = []
    all_class_ids = []

    start = time.perf_counter()

    with torch.inference_mode():

        for batch in tqdm(
            dataloader,
            desc="Extract embeddings"
        ):
            images = batch["image"].to(
                device,
                non_blocking=True
            )

            features = model(images)

            features = torch.flatten(
                features,
                start_dim=1
            )

            features = l2_normalize(
                features
            )

            all_embeddings.append(
                features.cpu().numpy()
            )

            all_image_ids.extend(
                batch["image_id"].numpy()
            )

            all_class_ids.extend(
                batch["class_id"].numpy()
            )

    elapsed = time.perf_counter() - start

    embeddings = np.concatenate(
        all_embeddings,
        axis=0
    ).astype(np.float32)

    return (
        embeddings,
        np.asarray(all_image_ids),
        np.asarray(all_class_ids),
        elapsed
    )

## Cell 16 — Offline: extract gallery embeddings

In [ ]:
if DEVICE == "cuda":
    torch.cuda.synchronize()

gallery_embeddings, gallery_image_ids, gallery_class_ids, gallery_extract_time = \
    extract_embeddings(
        encoder,
        gallery_loader,
        DEVICE
    )

if DEVICE == "cuda":
    torch.cuda.synchronize()

print("Embedding shape:", gallery_embeddings.shape)
print("Extraction time:", gallery_extract_time, "sec")
print("dtype:", gallery_embeddings.dtype)

## Cell 17 — Validate gallery embeddings

In [ ]:
assert gallery_embeddings.ndim == 2
assert gallery_embeddings.shape[0] == len(gallery_df)
assert gallery_embeddings.shape[1] == 2048

embedding_norms = np.linalg.norm(
    gallery_embeddings,
    axis=1
)

print(
    "Norm min:",
    embedding_norms.min()
)

print(
    "Norm max:",
    embedding_norms.max()
)

print(
    "Norm mean:",
    embedding_norms.mean()
)

assert np.allclose(
    embedding_norms,
    1.0,
    atol=1e-4
)

print("✓ Gallery embeddings are L2 normalized.")

## Cell 18 — Lưu offline artifacts

In [ ]:
np.save(
    EMBEDDING_FILE,
    gallery_embeddings
)

gallery_meta = gallery_df.copy()

gallery_meta.to_csv(
    GALLERY_META_FILE,
    index=False
)

print("Embedding saved:", EMBEDDING_FILE)
print("Metadata saved :", GALLERY_META_FILE)

## Cell 19 — Đo memory của embedding

In [ ]:
embedding_memory_bytes = (
    gallery_embeddings.nbytes
)

embedding_memory_mb = (
    embedding_memory_bytes
    / (1024 ** 2)
)

print(
    f"Gallery embedding memory: "
    f"{embedding_memory_mb:.2f} MB"
)

print(
    f"Per vector: "
    f"{gallery_embeddings.shape[1] * 4 / 1024:.2f} KB"
)

## Cell 20 — Exact cosine search

Vì tất cả embedding đã được L2 normalize:

```text
cosine(q, x)
=
q · x
```

Do đó exact cosine search có thể thực hiện bằng matrix multiplication:

```text
Q [Nq × D]
×
Gᵀ [D × Ng]
=
S [Nq × Ng]
```

Sau đó lấy Top-K similarity lớn nhất.

Đây là **exact search**, không dùng ANN.

In [ ]:
# ==============================================================================
# KAGGLE OUTPUT & RESULTS PREVIEW BLOCK (INDEPENDENT BLOCK)
# ==============================================================================
import os
import shutil
from pathlib import Path
import pandas as pd

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if IS_KAGGLE:
    print("--- Kaggle Output & Preview ---")
    
    # 1. Print preview
    print("Preview of splits and gallery metadata:")
    gallery_file = Path("data/sampled/baseline_gallery.csv")
    if gallery_file.exists():
        print(f"Gallery split shape: {pd.read_csv(gallery_file).shape}")
    query_file = Path("data/sampled/baseline_query.csv")
    if query_file.exists():
        print(f"Query split shape: {pd.read_csv(query_file).shape}")
    meta_file = Path("outputs/baseline/gallery_metadata.csv")
    if meta_file.exists():
        print(f"Gallery metadata shape: {pd.read_csv(meta_file).shape}")
        
    # 2. Export to /kaggle/working
    print("\nCopying results to Kaggle output directory (/kaggle/working)...")
    for folder in ["outputs/baseline", "data/sampled"]:
        dest = Path("/kaggle/working") / folder
        dest.mkdir(parents=True, exist_ok=True)
        src = Path(folder)
        if src.exists():
            for item in src.iterdir():
                if item.is_file():
                    shutil.copy(item, dest)
                    print(f"Copied {item.name} to {dest}")
